# Approved R2: official treated-input audit and split
User approved R2 on 30 October 2026; supervisor sign-off remains unverified. Original audit stays blocked. Use this notebook instead of running the historical runner through all phases. Stop after Phase 2. Raw sources are copied locally on Drive, never redownloaded. No training/evaluation here.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import subprocess, sys, os, json
PROJECT_DIR = Path("/content/ai-image-fft-detector")
assert PROJECT_DIR.is_dir(), "Use the existing checkout with the original acquisition"
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=PROJECT_DIR, check=True)
os.chdir(PROJECT_DIR)
CONFIG_PATH = PROJECT_DIR / "config_r2.yaml"
assert CONFIG_PATH.is_file()
print(CONFIG_PATH.read_text())

## Official R2 audit
This remeasures originals, copies and verifies the existing cohort to a separate root, and measures the shared R2 treatment. It may take several minutes. The old audit is preserved. Completed R2 audits are not overwritten. An interrupted, provenance-matching run may resume; do not delete a completed record to rerun it.

In [ ]:
subprocess.run([sys.executable, "-u", "scripts/run_audit.py", "--config", str(CONFIG_PATH)], cwd=PROJECT_DIR, check=True)

In [ ]:
import yaml
settings = yaml.safe_load(CONFIG_PATH.read_text())
DATA_ROOT = Path(settings["paths"]["data_root"])
result = json.loads((DATA_ROOT / "audit/audit_result.json").read_text())
print(json.dumps(result, indent=2))
assert result["protocol"] == "R2" and result["audit_domain"] == "treated_input"
assert result["raw_audit_status"] == "blocked"
assert result["status"] == "passed" and result["final_probe"]["max_accuracy"] <= 0.60
assert result["mixed_label_groups"] == 0

## Create immutable R2 splits after passing audit review
The user approved R2 and continued Phase 2 execution. Review the explicit narrower audit domain and limitations above before continuing. No later phase is authorized by this cell.

In [ ]:
subprocess.run([sys.executable, "-u", "scripts/make_splits.py", "--config", str(CONFIG_PATH)], cwd=PROJECT_DIR, check=True)

In [ ]:
import pandas as pd
splits = pd.read_csv(DATA_ROOT / "splits.csv", dtype={"id": str,"group_id": str})
counts = splits.groupby(["split","label"]).size().unstack(fill_value=0)
assert counts[0].eq(counts[1]).all()
assert splits.groupby("group_id")["split"].nunique().eq(1).all()
print(counts)
print((PROJECT_DIR / settings["paths"]["artifacts_root"] / "splits.sha256").read_text())

STOP after Phase 2. Preserve and review R2 reports and split provenance. Use config_r2.yaml explicitly for future preprocessing/extraction/training commands. The old runner still targets the old experiment and must not be run through all cells. Test features remain sealed. No model, practical target, freeze or test evaluation is supplied here.